# L06 | From a perceptron to a multilayer network
**CH5330 Statistics — ML teaching development v0.3**  
**Student notebook · self-contained · no GPU required**

This notebook can be used in the lecture **or independently afterward**. Read the explanation above a cell, predict the outcome, then run the cell with **Shift+Enter**. The main path works without filling in the optional exercises.

## The problem is familiar, but the model changes

Use the same synthetic two-moons system as L05. We will replace hand-engineered quadratic features with nonlinear hidden units.

**Before you run:** Why is a single straight decision boundary inadequate for these data?

**Lecture mapping:** L06 — weighted sums, activations, feed-forward architecture, function approximation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# A fixed random seed gives the same synthetic data whenever you restart.
SEED = 42
rng = np.random.default_rng(SEED)

def make_two_moons(n_per_class=100, noise=0.16, seed=SEED):
    """Make two interlocking crescent-shaped groups of 2D points.

    This is a deliberately simple alternative to importing scikit-learn.
    The labels are 0 (upper crescent) and 1 (lower crescent).
    """
    random = np.random.default_rng(seed)
    t0 = random.uniform(0, np.pi, n_per_class)
    t1 = random.uniform(0, np.pi, n_per_class)
    upper = np.column_stack([np.cos(t0), np.sin(t0)])
    lower = np.column_stack([1 - np.cos(t1), 0.5 - np.sin(t1)])
    X = np.vstack([upper, lower])
    X += random.normal(0, noise, X.shape)
    y = np.array([0] * n_per_class + [1] * n_per_class)
    # Shuffle the data so that the two classes are interspersed.
    order = random.permutation(len(y))
    return X[order], y[order]

def split_data(X, y, seed=SEED):
    """Keep a test set separate. Only training points fit the model."""
    random = np.random.default_rng(seed + 1)
    # Split *within each class* so that small validation/test sets stay balanced.
    by_class = [random.permutation(np.where(y == label)[0]) for label in (0, 1)]
    train_ids, valid_ids, test_ids = [], [], []
    for indices in by_class:
        n_train = int(0.70 * len(indices))
        n_valid = int(0.15 * len(indices))
        train_ids.extend(indices[:n_train])
        valid_ids.extend(indices[n_train:n_train + n_valid])
        test_ids.extend(indices[n_train + n_valid:])
    return [(X[ids], y[ids]) for ids in (train_ids, valid_ids, test_ids)]

def show_points(X, y, title='Two moons'):
    plt.figure(figsize=(6, 4))
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=21, edgecolors='none')
    plt.xlabel('$x_1$'); plt.ylabel('$x_2$'); plt.title(title)
    plt.axis('equal'); plt.grid(alpha=0.15); plt.show()

X, y = make_two_moons()
(X_train, y_train), (X_val, y_val), (X_test, y_test) = split_data(X, y)
print(f'Training: {len(y_train)} | Validation: {len(y_val)} | Test: {len(y_test)}')
show_points(X_train, y_train, 'Training data (two interleaving crescents)')

In [ ]:
def sigmoid(z):
    """Logistic activation: maps any real number to (0, 1)."""
    z = np.clip(z, -40, 40)  # prevents overflow in exp for extreme inputs
    return 1 / (1 + np.exp(-z))

def binary_cross_entropy(y, probabilities):
    """Average loss for labels 0/1; clipping avoids log(0)."""
    p = np.clip(probabilities, 1e-10, 1 - 1e-10)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def accuracy(y, probabilities):
    predictions = (probabilities >= 0.5).astype(int)
    return np.mean(predictions == y)

def show_boundary(predict_probabilities, X, y, title):
    """Draw the p(class 1)=0.5 boundary of any binary classifier."""
    xmin, xmax = X[:, 0].min() - 0.3, X[:, 0].max() + 0.3
    ymin, ymax = X[:, 1].min() - 0.3, X[:, 1].max() + 0.3
    xs = np.linspace(xmin, xmax, 90)
    ys = np.linspace(ymin, ymax, 90)
    xx, yy = np.meshgrid(xs, ys)
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    probs = predict_probabilities(grid).reshape(xx.shape)
    plt.figure(figsize=(6, 4))
    plt.contourf(xx, yy, probs, levels=np.linspace(0, 1, 11), cmap='coolwarm', alpha=0.32)
    plt.contour(xx, yy, probs, levels=[0.5], colors='black', linewidths=1.3)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=17, edgecolors='none')
    plt.xlabel('$x_1$'); plt.ylabel('$x_2$'); plt.title(title)
    plt.axis('equal'); plt.show()

## One neuron: the basic calculation

$$z=w_1x_1+w_2x_2+b,\qquad a=\sigma(z).$$

A *neuron* is a convenient name for this elementary operation; we do not need a biological analogy. A nonlinear activation makes the value $a$ a nonlinear function of the weighted sum $z$.

In [ ]:
x = np.array([1.0, -1.0])
w = np.array([0.5, 1.0])
b = 0.0
z = x @ w + b
a = sigmoid(z)
print(f'Weighted sum z = {z:.4f}; activated output a = {a:.4f}')
# OPTIONAL: change b to +1 and predict how a changes.

## Why nonlinearity between layers matters

Consider two layers with **no nonlinear activation**:

$$\mathbf h=W_1\mathbf x+\mathbf b_1,\qquad \hat y=W_2\mathbf h+b_2.$$

Substituting the first into the second gives

$$\hat y=(W_2W_1)\mathbf x+(W_2\mathbf b_1+b_2).$$

This is still an affine function of the input. Increasing the number of *purely linear* layers does not create a nonlinear decision boundary.

In [ ]:
# Verify that two linear layers collapse to one affine transformation.
W1 = np.array([[1.0, -0.4], [0.6, 0.8]])
b1 = np.array([0.2, -0.3])
W2 = np.array([[0.5, 1.2]])
b2 = 0.1
x = np.array([0.4, -0.8])
layer_by_layer = (W2 @ (W1 @ x + b1))[0] + b2
collapsed = ((W2 @ W1) @ x)[0] + (W2 @ b1)[0] + b2
print('Layer-by-layer:', round(layer_by_layer, 6))
print('Collapsed form: ', round(collapsed, 6))
assert np.isclose(layer_by_layer, collapsed)

## Build a small MLP (2 inputs → hidden units → 1 output)

Our network first computes hidden activations, then an output probability. You can choose a **tanh** hidden activation or the **identity** activation for a controlled comparison:

$$H=\phi(XW_1+b_1),\qquad p=\sigma(HW_2+b_2),$$

where $\phi(z)=\tanh(z)$ (nonlinear) or $\phi(z)=z$ (identity). The function $\phi$ acts separately on every element of the hidden-layer array. The final sigmoid acts on the one output score per observation.

For a dataset with $N$ observations and $m$ hidden units, the array sizes are:

- $X$: $(N,2)$; $W_1$: $(2,m)$; $b_1$: $(m,)$.
- $H$: $(N,m)$; $W_2$: $(m,1)$; $b_2$: $(1,)$.

The code below includes training so that L06 can focus on **representation and architecture**. The derivatives used by the training function will be developed explicitly in L07.

In [ ]:
def create_network(hidden_units=8, seed=SEED):
    """Initialise weights and biases for a 2 -> m -> 1 network."""
    random = np.random.default_rng(seed)
    return {
        'W1': random.normal(0, 0.8, size=(2, hidden_units)),
        'b1': np.zeros(hidden_units),
        'W2': random.normal(0, 0.4, size=(hidden_units, 1)),
        'b2': np.zeros(1),
    }

def hidden_activation(Z, activation='tanh'):
    """Apply one selected hidden activation elementwise."""
    if activation == 'tanh':
        return np.tanh(Z)
    if activation == 'identity':
        return Z
    raise ValueError("activation must be 'tanh' or 'identity'")

def hidden_derivative(H, activation='tanh'):
    """Derivative of hidden activation, expressed using hidden output H."""
    if activation == 'tanh':
        return 1 - H**2
    if activation == 'identity':
        return np.ones_like(H)
    raise ValueError("activation must be 'tanh' or 'identity'")

def forward(X, net, activation='tanh'):
    """Return hidden features H and final sigmoid probabilities."""
    H = hidden_activation(X @ net['W1'] + net['b1'], activation)
    p = sigmoid(H @ net['W2'] + net['b2']).ravel()
    return H, p

def train_network(X, y, hidden_units=8, steps=2200, learning_rate=0.7,
                  seed=SEED, activation='tanh'):
    """Train with BCE; L07 explains the chain rule for these gradients."""
    if activation not in ('tanh', 'identity'):
        raise ValueError("activation must be 'tanh' or 'identity'")
    net = create_network(hidden_units, seed)
    losses = []
    for step in range(steps):
        H, p = forward(X, net, activation=activation)
        if step % 50 == 0:
            losses.append(binary_cross_entropy(y, p))
        dZ2 = (p - y)[:, None] / len(y)
        dW2 = H.T @ dZ2
        db2 = dZ2.sum(axis=0)
        dH = dZ2 @ net['W2'].T
        dZ1 = dH * hidden_derivative(H, activation=activation)
        dW1 = X.T @ dZ1
        db1 = dZ1.sum(axis=0)
        net['W1'] -= learning_rate * dW1
        net['b1'] -= learning_rate * db1
        net['W2'] -= learning_rate * dW2
        net['b2'] -= learning_rate * db2
    return net, losses

net = create_network(hidden_units=4)
H, p = forward(X_train, net)
print('Input X shape:', X_train.shape)
print('Hidden activation shape:', H.shape)
print('Output probability shape:', p.shape)
print('First three probabilities:', np.round(p[:3], 3))


## Checkpoint B — change the architecture

**Predict first:** What do you expect when the number of hidden units increases from 1 to 8? There is no guarantee that a bigger network trains better in every run.

Run the next cell with `activation = "tanh"` first, then change `hidden_units`. Keep the training data, seed, learning rate and number of steps fixed so that the comparison is interpretable.

In [ ]:
hidden_units = 8  # ACTIVITY: compare 1, 2, 4, 8, 16
activation = 'tanh'  # keep tanh for the representation-learning experiment below
net, losses = train_network(X_train, y_train, hidden_units=hidden_units,
                            activation=activation)
predict_mlp = lambda X: forward(X, net, activation=activation)[1]
show_boundary(predict_mlp, X_train, y_train,
              f'{hidden_units} hidden units ({activation}): learned decision boundary')
print('Training accuracy:', round(accuracy(y_train, predict_mlp(X_train)), 3))
print('Validation accuracy:', round(accuracy(y_val, predict_mlp(X_val)), 3))
plt.plot(np.arange(len(losses)) * 50, losses)
plt.xlabel('Gradient-descent step'); plt.ylabel('Training BCE loss'); plt.show()


## Checkpoint C — experimental control: identity versus tanh

**Question:** Does increasing the number of hidden units make the decision boundary nonlinear *without* a nonlinear activation?

Choose the hidden activation in the next cell:

- `identity`: $\phi(z)=z$ — no nonlinearity. Any number of affine layers collapses to one affine score.
- `tanh`: $\phi(z)=\tanh(z)$ — nonlinear learned features can support a curved decision boundary.

Both models use **the same** training data, split, initialisation seed, hidden width, learning rate, and number of steps. We vary only the activation rule and use its correct derivative in training. Both models are genuinely retrained; changing the activation *after* training would not constitute a controlled comparison.

**Predict before running:** Will the `identity` boundary remain straight even when `hidden_units = 16`? Will `tanh` necessarily have a higher validation accuracy in every run?

The validation set is used for the experiment; keep the held-out **test set** untouched for the later final evaluation.


In [ ]:
# ACTIVITY: change the hidden width (e.g., 2, 8, 16) and compare activations.
control_hidden_units = 8
control_steps = 2200
control_learning_rate = 0.7
control_seed = SEED

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
control_results = {}
for ax, act in zip(axes, ('identity', 'tanh')):
    fitted, history = train_network(
        X_train, y_train, hidden_units=control_hidden_units,
        steps=control_steps, learning_rate=control_learning_rate,
        seed=control_seed, activation=act
    )
    predict = lambda X, fitted=fitted, act=act: forward(X, fitted, activation=act)[1]
    xlo, xhi = X_train[:, 0].min() - 0.3, X_train[:, 0].max() + 0.3
    ylo, yhi = X_train[:, 1].min() - 0.3, X_train[:, 1].max() + 0.3
    xx, yy = np.meshgrid(np.linspace(xlo, xhi, 100), np.linspace(ylo, yhi, 100))
    pp = predict(np.column_stack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
    ax.contourf(xx, yy, pp, levels=np.linspace(0, 1, 11),
                cmap='coolwarm', alpha=0.3)
    ax.contour(xx, yy, pp, levels=[0.5], colors='black', linewidths=1.4)
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train,
               cmap='coolwarm', vmin=0, vmax=1, s=14, edgecolors='none')
    ax.set(xlabel='$x_1$', ylabel='$x_2$', title=f'{act} · {control_hidden_units} hidden units')
    ax.set_aspect('equal', adjustable='box')
    control_results[act] = {
        'train_acc': accuracy(y_train, predict(X_train)),
        'valid_acc': accuracy(y_val, predict(X_val)),
        'final_loss': binary_cross_entropy(y_train, predict(X_train)),
    }
plt.tight_layout()
plt.show()

for act, metrics in control_results.items():
    print(f"{act:8s} | train={metrics['train_acc']:.3f} "
          f"| validation={metrics['valid_acc']:.3f} "
          f"| training BCE={metrics['final_loss']:.4f}")


**Interpretation checkpoint.** The identity network is still a function of the form
$s(\mathbf x)=\tilde{\mathbf w}^{\mathsf T}\mathbf x+\tilde b$,
followed by sigmoid, regardless of its hidden width. Hence its $p=0.5$
boundary is a straight line (unless a degenerate score produces no unique boundary).

For `tanh`, the score is nonlinear in the original inputs, so the boundary **can** curve.
The graph and validation accuracy tell us how well the trained solution fits this dataset;
a curved boundary alone is **not** evidence of better generalisation.

Return to **Checkpoint B** with `activation = "tanh"` for the subsequent hidden-feature
visualisation: it deliberately reuses the `net` fitted there, rather than the separate
control networks above.


## What does universal approximation mean?

With a suitable nonlinear activation, a sufficiently wide feed-forward network can approximate any **continuous real-valued function on a compact domain**, to arbitrary accuracy, in the sense specified by the theorem.

This is an *existence statement*, **not** a guarantee that an optimiser finds the parameters, that the required network is small, or that test error will be low. Approximating a smooth real-valued function and classifying noisy observations are related but distinct tasks.

**Exit question:** Why would doubling the number of *linear* layers fail to solve two moons, while adding a nonlinear hidden layer can help?

## Optional extension — inspect activations

The hidden-layer outputs are nonlinear features learned by the network. Their values differ from point to point. Inspect them before attempting to code the forward pass yourself.

In [ ]:
H, p = forward(X_train, net)
plt.figure(figsize=(6, 4))
plt.scatter(H[:, 0], H[:, 1] if H.shape[1] > 1 else np.zeros(len(H)),
            c=y_train, cmap='coolwarm', s=20)
plt.xlabel('Hidden activation 1')
plt.ylabel('Hidden activation 2 (0 if only one unit)')
plt.title('Two learned hidden features')
plt.show()

## Explore — representation learning: change coordinates, simplify the decision

The previous graph shows two hidden activations. **They are learned features**, not the original $x_1,x_2$ coordinates.

For our trained network, every input point $\mathbf{x}\in\mathbb{R}^2$ is mapped to

$$\mathbf{h}(\mathbf{x})=\tanh(\mathbf{x}W_1+\mathbf{b}_1)\in\mathbb{R}^{m}.$$

The network then calculates a **score** (also called a *logit*):

$$s(\mathbf{x})=\mathbf{h}(\mathbf{x})W_2+b_2,\qquad p(\mathbf{x})=\sigma(s(\mathbf{x})).$$

Since the sigmoid is increasing, $p\geq 0.5$ **exactly when** $s\geq0$. Thus the final classifier uses a *linear decision boundary in the full hidden-feature space*, although the corresponding boundary in the original $x_1,x_2$ plane can be curved.

**Important distinction:** $\mathbf{x}\mapsto\mathbf{h}(\mathbf{x})$ is a learned **nonlinear mapping**, not simply a linear projection. Plotting $(h_1,h_2)$ shows only **two coordinates**. For $m>2$, the classes may be separable using all $m$ features even if the two-dimensional scatter looks mixed.

**Predict before running:** Which of the following should reveal the actual network decision most clearly: the original moons, the first two hidden activations, or the final score $s$?

In [ ]:
# Three views of exactly the SAME trained network and training observations.
# Re-use "net" trained in Checkpoint B above; no retraining happens here.
H_train, p_train = forward(X_train, net)
score_train = (H_train @ net['W2'] + net['b2']).ravel()
class_pred = (score_train >= 0).astype(int)

print(f'Input dimensions: {X_train.shape[1]}')
print(f'Hidden-feature dimensions: {H_train.shape[1]}')
print(f'Training accuracy from logit sign: {np.mean(class_pred == y_train):.3f}')
# The final score and sigmoid probability lead to identical class predictions.
assert np.array_equal(class_pred, (p_train >= 0.5).astype(int))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# View 1: original two-moons input coordinates.
axes[0].scatter(X_train[:, 0], X_train[:, 1], c=y_train,
                cmap='coolwarm', s=19, vmin=0, vmax=1)
axes[0].set(xlabel='$x_1$', ylabel='$x_2$', title='1. Original input space')
axes[0].axis('equal')

# View 2: only two of the m learned hidden coordinates.
# When m == 1, display the single feature on the horizontal axis.
h_second = H_train[:, 1] if H_train.shape[1] >= 2 else np.zeros(len(H_train))
axes[1].scatter(H_train[:, 0], h_second, c=y_train,
                cmap='coolwarm', s=19, vmin=0, vmax=1)
axes[1].set(xlabel='$h_1$', ylabel='$h_2$' if H_train.shape[1] >= 2 else '0',
            title='2. First two learned features')

# View 3: all hidden coordinates combined by the actual output weights.
# Vertical offsets are a VISUAL aid only; they do not enter the network.
visual_offset = np.where(y_train == 0, -0.08, 0.08)
axes[2].scatter(score_train, visual_offset, c=y_train,
                cmap='coolwarm', s=19, vmin=0, vmax=1)
axes[2].axvline(0, color='black', linestyle='--', label='Decision threshold $s=0$')
axes[2].set(xlabel='Final logit $s = hW_2+b_2$', ylabel='Visual offset (not a feature)',
            title='3. One score from ALL hidden features')
axes[2].set_yticks([])
axes[2].legend(fontsize=8, loc='upper center')

plt.tight_layout()
plt.show()

### Investigate the representation, not just the boundary

The score above combines **all** hidden features. Its threshold at $s=0$ is a point on a one-dimensional axis (the equivalent of a hyperplane in the full $m$-dimensional hidden space). Class overlap in the score plot corresponds to misclassification on these training observations; overlap in $(h_1,h_2)$ alone need not.

Try the following while keeping the data split and seed fixed:

1. Return to **Checkpoint B**, change `hidden_units` from `8` to `2`, run the training cell, then rerun the three-view cell. What changes in the hidden representation?
2. Repeat with `hidden_units = 8` or `16`. Is separation in the $(h_1,h_2)$ view necessary for successful classification by the full model?
3. Compare training and validation accuracy. Does a visually cleaner training representation guarantee better generalisation?

**Takeaway:** The hidden layer can *learn a nonlinear representation* in which the **last layer applies a simple linear decision**. The training loss determines which representation is useful; it does not guarantee perfect separation.

**Lecture mapping:** $\hat y=a+bx\ \to\ \phi(a+bx)\ \to\ \sum_j v_j\phi(a_j+b_jx)+c$. The final expression is a weighted combination of learned nonlinear features.

In [ ]:
# OPTIONAL: report a small numerical summary of score separation.
# This is descriptive, not a formal hypothesis test.
for label in (0, 1):
    values = score_train[y_train == label]
    print(f'Class {label}: mean logit = {values.mean():+.3f}, '
          f'range = [{values.min():+.3f}, {values.max():+.3f}]')

errors = np.where(class_pred != y_train)[0]
print(f'Misclassified training points: {len(errors)} / {len(y_train)}')
